# Evaluación de los modelos

- **E2:** mAP@0.5 y mAP@0.5:0.95 del modelo principal sobre `full_test` (cámaras nunca vistas), global y por especie, y efecto del desbalance entre especies.
- **E3:** generalización entre cámaras con la partición oficial ECCV: desempeño en cámaras vistas (`eccv_cis_test`) frente a cámaras nuevas (`eccv_trans_test`).

Las métricas se calculan con `src/eval/evaluate.py` y quedan en `results/` (CSV) y las figuras en `reports/figures/`.

## Preparar la máquina

Igual que en el entrenamiento: se ejecuta `colab_setup.ipynb` completo (extracción de imágenes y etiquetas YOLO).

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')
REPO_DIR = '/content/redes-parcial-p3'
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/Roxana-Vargas/redes-parcial-p3.git {REPO_DIR}
%cd {REPO_DIR}
!git pull
%run notebooks/colab_setup.ipynb

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

PROJECT = '/content/drive/MyDrive/redes-parcial-p3/runs'
FULL = f'{PROJECT}/full_yolov8s/weights/best.pt'
ECCV = f'{PROJECT}/eccv_yolov8s/weights/best.pt'

BLUE, ORANGE = '#2a78d6', '#eb6834'
INK, INK_2, GRID = '#0b0b0b', '#52514e', '#e5e4e0'
plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 200, 'savefig.bbox': 'tight',
    'font.size': 10, 'axes.titlesize': 11, 'axes.titleweight': 'bold',
    'axes.edgecolor': GRID, 'axes.labelcolor': INK_2, 'axes.titlecolor': INK,
    'xtick.color': INK_2, 'ytick.color': INK_2, 'text.color': INK,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.6,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.axisbelow': True,
})

## E2 · Modelo principal en cámaras nunca vistas (`full_test`)

In [ ]:
!python src/eval/evaluate.py --weights {FULL} --data data/yolo/data_full.yaml --split full_test --model full_yolov8s

In [ ]:
e2 = pd.read_csv('results/eval_full_yolov8s_full_test.csv').sort_values('cajas_train', ascending=False)
display(e2.reset_index(drop=True))

### Efecto del desbalance

AP@0.5 de cada especie según cuántas cajas tenía en entrenamiento (escala logarítmica). Si el desbalance pesa, las especies con pocos ejemplos quedan abajo a la izquierda.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.scatter(e2['cajas_train'], e2['AP50'], s=40, color=BLUE, edgecolor='white', linewidth=1, zorder=3)
for _, r in e2.iterrows():
    ax.annotate(r['especie'], (r['cajas_train'], r['AP50']), textcoords='offset points', xytext=(5, 3),
                fontsize=8, color=INK_2)
ax.set_xscale('log')
ax.set_ylim(0, 1.02)
ax.set_xlabel('Cajas de la especie en entrenamiento (escala log)')
ax.set_ylabel('AP@0.5 en full_test')
ax.set_title('AP por especie frente a ejemplos de entrenamiento')
fig.savefig('reports/figures/e2_ap_vs_ejemplos.png')
plt.show()

corr = e2[['cajas_train', 'AP50']].corr(method='spearman').iloc[0, 1]
print(f'Correlación de Spearman entre nº de cajas en train y AP@0.5: {corr:.2f}')

## E3 · Generalización entre cámaras (partición oficial ECCV)

Se evalúan los dos modelos en las mismas dos particiones:

- `eccv_cis_test`: fotos nuevas de las 10 cámaras de `eccv_train` (cámaras vistas).
- `eccv_trans_test`: fotos de 9 cámaras que ninguno de los dos modelos vio al entrenar.

**Brecha = mAP(cis) − mAP(trans).** El modelo E3 responde a lo que pide el enunciado; el modelo principal permite ver si entrenar con más cámaras (76 en vez de 10) reduce la brecha.

In [ ]:
for model, weights, data in [('eccv_yolov8s', ECCV, 'data/yolo/data_eccv.yaml'),
                             ('full_yolov8s', FULL, 'data/yolo/data_full.yaml')]:
    for split in ['eccv_cis_test', 'eccv_trans_test']:
        !python src/eval/evaluate.py --weights {weights} --data {data} --split {split} --model {model}

In [ ]:
summary = pd.read_csv('results/eval_summary.csv')
e3 = summary[summary['particion'].isin(['eccv_cis_test', 'eccv_trans_test'])]
table = e3.pivot(index='modelo', columns='particion', values=['mAP50', 'mAP50_95'])
for metric in ['mAP50', 'mAP50_95']:
    table[(metric, 'brecha')] = table[(metric, 'eccv_cis_test')] - table[(metric, 'eccv_trans_test')]
display(table.round(3))

In [ ]:
models = ['eccv_yolov8s', 'full_yolov8s']
labels = {'eccv_yolov8s': 'Modelo E3\n(10 cámaras)', 'full_yolov8s': 'Modelo principal\n(76 cámaras)'}
x = range(len(models))
cis = [table.loc[m, ('mAP50', 'eccv_cis_test')] for m in models]
trans = [table.loc[m, ('mAP50', 'eccv_trans_test')] for m in models]

fig, ax = plt.subplots(figsize=(6, 4))
w = 0.36
ax.bar([i - w / 2 for i in x], cis, width=w, color=BLUE, label='cis (cámaras vistas)')
ax.bar([i + w / 2 for i in x], trans, width=w, color=ORANGE, label='trans (cámaras nuevas)')
for i, (c, t) in enumerate(zip(cis, trans)):
    ax.text(i - w / 2, c + 0.01, f'{c:.2f}', ha='center', fontsize=8, color=INK_2)
    ax.text(i + w / 2, t + 0.01, f'{t:.2f}', ha='center', fontsize=8, color=INK_2)
ax.set_xticks(list(x), [labels[m] for m in models])
ax.set_ylim(0, 1.05)
ax.set_ylabel('mAP@0.5')
ax.set_title('Cámaras vistas frente a cámaras nuevas')
ax.grid(axis='x', visible=False)
ax.legend(frameon=False, loc='upper right')
fig.savefig('reports/figures/e3_cis_vs_trans.png')
plt.show()

### Brecha por especie (modelo E3)

Especies en las que más cae el AP al pasar de cámaras vistas a cámaras nuevas.

In [ ]:
cis_sp = pd.read_csv('results/eval_eccv_yolov8s_eccv_cis_test.csv').set_index('especie')
trans_sp = pd.read_csv('results/eval_eccv_yolov8s_eccv_trans_test.csv').set_index('especie')
gap = pd.DataFrame({'AP50_cis': cis_sp['AP50'], 'AP50_trans': trans_sp['AP50']}).dropna()
gap['brecha'] = gap['AP50_cis'] - gap['AP50_trans']
display(gap.sort_values('brecha', ascending=False).round(3))

## Guardar resultados

Los CSV de `results/` y las figuras de `reports/figures/` se generan en la máquina de Colab. Para llevarlos al repo se copian a Drive y desde ahí a la PC.

In [ ]:
!mkdir -p /content/drive/MyDrive/redes-parcial-p3/resultados
!cp results/eval_*.csv /content/drive/MyDrive/redes-parcial-p3/resultados/
!cp reports/figures/e2_*.png reports/figures/e3_*.png /content/drive/MyDrive/redes-parcial-p3/resultados/
!ls /content/drive/MyDrive/redes-parcial-p3/resultados